# Model 7: Girls Reintegration Readiness (CRISP-DM)

This notebook implements an end-to-end **machine learning pipeline** for estimating resident reintegration readiness, following **CRISP-DM** (Cross-Industry Standard Process for Data Mining): Business Understanding → Data Understanding → Data Preparation → Modeling → Evaluation → Deployment.

**Alignment:** The steps below mirror the logic in `backend/ML/girls_reintegration_train.py` (production train script) as of project development; this file is **self-contained** for grading—no imports from `backend/ML`.

**How to run:** Open this notebook from the repo with **working directory = project root** *or* `ml-pipelines/` (the data path logic tries both). Use **Run All**.


## Phase 1: Business Understanding

**Objective:** Predict **readiness for reintegration** (binary supervised label derived from `reintegration_status` text) so case managers can prioritize residents on a worklist.

**Stakeholders:** Admin leadership, social workers, safehouse teams.

**Predictive vs explanatory:**
- **Predictive:** Random Forest classifier scores each resident (probability of readiness); used for ranking.
- **Explanatory companion:** Logistic regression on the same features for interpretable direction of associations (not causal claims).

**Success metrics (held-out test):** ROC-AUC, Average Precision, F1 vs a **majority-class baseline**.

**Constraints:** The model is **associational**—feature importance does not prove that changing a feature causes a different outcome.


## Phase 2: Data Understanding

**Sources (JSON payload shape, same as API stdin to the production script):** `residents`, `safehouses`, `process_recordings`, `home_visitations`, `education_records`, `health_records`, `intervention_plans`, `incident_reports`.

**Label:** `readiness_label` maps free-text `reintegration_status` to 0/1; unmapped statuses are excluded from training but can still appear in scoring for all residents.

If `ml-pipelines/data/girls_reintegration_export.json` is absent, we generate a **synthetic** payload so **Run All** works without a database.


In [ ]:
# --- Imports & paths ---
from __future__ import annotations

import json
import re
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

# Resolve export path: repo root or ml-pipelines as cwd
_CWD = Path.cwd()
_CANDIDATES = [
    _CWD / "ml-pipelines" / "data" / "girls_reintegration_export.json",
    _CWD / "data" / "girls_reintegration_export.json",
]
EXPORT_PATH = next((p for p in _CANDIDATES if p.exists()), None)
print("Export file:", EXPORT_PATH if EXPORT_PATH else "(not found - will use synthetic data)")



In [ ]:
# --- Label & parsing helpers (same rules as production script) ---

def safe_float(v, default=0.0):
    try:
        return float(v)
    except Exception:
        return default


def parse_numeric_from_text(v, default=0.0):
    if v is None:
        return default
    if isinstance(v, (int, float)):
        return float(v)
    text = str(v)
    digits = "".join(ch for ch in text if ch.isdigit() or ch == ".")
    if not digits:
        return default
    try:
        return float(digits)
    except Exception:
        return default


def normalize_status(v: str) -> str:
    if v is None:
        return ""
    return str(v).strip().lower()


def readiness_label(status: str):
    # Map free-text reintegration_status to 0/1. Unmapped values -> None (excluded from training).
    # Kept in sync with backend/ML/girls_reintegration_train.py
    s = normalize_status(status)
    if not s:
        return None
    negative_phrases = (
        "high risk",
        "not ready",
        "not completed",
        "in progress",
        "pending review",
        "on hold",
        "on-hold",
        "preparation",
        "awaiting",
        "screening",
        "intake",
    )
    for phrase in negative_phrases:
        if phrase in s:
            return 0
    positive_phrases = (
        "successfully",
        "reintegration complete",
        "case closed",
        "closed successfully",
    )
    for phrase in positive_phrases:
        if phrase in s:
            return 1
    words = set(re.findall(r"[a-z0-9]+", s))
    positive_words = frozenset(
        {
            "ready",
            "readiness",
            "reintegrated",
            "reintegration",
            "successful",
            "success",
            "completed",
            "complete",
            "stable",
            "discharged",
            "graduated",
            "reunified",
            "restored",
            "transitioned",
            "transition",
            "closure",
            "closed",
            "achieved",
            "cleared",
            "exit",
            "reunification",
        }
    )
    negative_words = frozenset(
        {
            "pending",
            "reopened",
            "unsafe",
            "active",
            "ongoing",
            "enrolled",
            "current",
            "admitted",
            "withdrawn",
            "cancelled",
            "canceled",
            "suspended",
            "deferred",
            "delayed",
            "stalled",
            "paused",
            "monitoring",
            "hold",
            "waiting",
            "tbd",
            "unknown",
        }
    )
    if words & positive_words:
        return 1
    if words & negative_words:
        return 0
    return None


FEATURE_COLUMNS = [
    "safehouse_code", "case_status", "case_category", "current_risk_level", "initial_risk_level",
    "reintegration_type", "present_age_num", "length_of_stay_num", "process_recordings_count",
    "home_visitations_count", "intervention_plans_count", "incident_reports_count",
    "session_duration_minutes", "progress_noted", "concerns_flagged", "referral_made",
    "follow_up_needed", "safety_concerns_noted", "attendance_rate", "progress_percent",
    "gpa_like_score", "nutrition_score", "sleep_score", "energy_score", "general_health_score",
    "resolved", "high_severity",
]


def build_preprocessor(df: pd.DataFrame) -> ColumnTransformer:
    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    categorical_cols = [c for c in df.columns if c not in numeric_cols]
    return ColumnTransformer(
        [
            (
                "num",
                Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]),
                numeric_cols,
            ),
            (
                "cat",
                Pipeline(
                    [
                        ("imp", SimpleImputer(strategy="most_frequent")),
                        ("ohe", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_cols,
            ),
        ]
    )



In [ ]:
def make_synthetic_payload() -> dict:
    # Minimal JSON-shaped payload: enough labeled rows in two classes for stratified split.
    residents = []
    for i in range(10):
        residents.append(
            {
                "resident_id": i + 1,
                "internal_code": f"POS{i+1:03d}",
                "safehouse_id": 1,
                "case_status": "open",
                "case_category": "standard",
                "current_risk_level": "medium",
                "initial_risk_level": "high",
                "present_age": str(14 + (i % 5)),
                "reintegration_status": "Completed successfully",
                "reintegration_type": "family",
                "length_of_stay": f"{90 + i} days",
                "date_of_admission": "2023-06-01",
                "date_closed": None,
            }
        )
    for i in range(10):
        residents.append(
            {
                "resident_id": 20 + i,
                "internal_code": f"NEG{i+1:03d}",
                "safehouse_id": 1,
                "case_status": "open",
                "case_category": "standard",
                "current_risk_level": "high",
                "initial_risk_level": "high",
                "present_age": str(15 + (i % 4)),
                "reintegration_status": "Active in program",
                "reintegration_type": "guardian",
                "length_of_stay": f"{30 + i} days",
                "date_of_admission": "2024-01-15",
                "date_closed": None,
            }
        )
    safehouses = [{"safehouse_id": 1, "safehouse_code": "SH01", "name": "Demo Safehouse"}]
    process_recordings = []
    for rid in [1, 2, 3, 21, 22]:
        process_recordings.append(
            {
                "resident_id": rid,
                "session_duration_minutes": 45,
                "progress_noted": True,
                "concerns_flagged": False,
                "referral_made": rid % 2 == 0,
            }
        )
    home_visitations = [
        {"resident_id": 1, "follow_up_needed": False, "safety_concerns_noted": False},
        {"resident_id": 21, "follow_up_needed": True, "safety_concerns_noted": True},
    ]
    education_records = [
        {"resident_id": 1, "attendance_rate": 0.9, "progress_percent": 0.85, "gpa_like_score": 3.2},
        {"resident_id": 21, "attendance_rate": 0.6, "progress_percent": 0.5, "gpa_like_score": 2.5},
    ]
    health_records = [
        {"resident_id": 1, "nutrition_score": 4, "sleep_score": 4, "energy_score": 4, "general_health_score": 4},
        {"resident_id": 21, "nutrition_score": 3, "sleep_score": 2, "energy_score": 3, "general_health_score": 3},
    ]
    intervention_plans = [{"resident_id": 1, "status": "active", "plan_category": "edu"}]
    incident_reports = [
        {"resident_id": 21, "severity": "low", "resolved": True},
        {"resident_id": 22, "severity": "high", "resolved": False},
    ]
    return {
        "residents": residents,
        "safehouses": safehouses,
        "process_recordings": process_recordings,
        "home_visitations": home_visitations,
        "education_records": education_records,
        "health_records": health_records,
        "intervention_plans": intervention_plans,
        "incident_reports": incident_reports,
        "initiated_by": "notebook_synthetic",
    }


def load_payload() -> dict:
    if EXPORT_PATH is not None:
        with open(EXPORT_PATH, encoding="utf-8") as f:
            return json.load(f)
    return make_synthetic_payload()


payload = load_payload()
for k in ("residents", "safehouses", "process_recordings", "home_visitations", "education_records", "health_records", "intervention_plans", "incident_reports"):
    if k not in payload:
        payload[k] = [] if k != "initiated_by" else "notebook"
print("Payload keys:", list(payload.keys()))



In [ ]:
def build_feature_tables(payload: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    # Returns (base_all_residents_features, train_df_labeled); mirrors girls_reintegration_train.py.
    residents = pd.DataFrame(payload.get("residents", []))
    safehouses = pd.DataFrame(payload.get("safehouses", []))
    process_recordings = pd.DataFrame(payload.get("process_recordings", []))
    home_visitations = pd.DataFrame(payload.get("home_visitations", []))
    education_records = pd.DataFrame(payload.get("education_records", []))
    health_records = pd.DataFrame(payload.get("health_records", []))
    intervention_plans = pd.DataFrame(payload.get("intervention_plans", []))
    incident_reports = pd.DataFrame(payload.get("incident_reports", []))

    if residents.empty:
        raise ValueError("Need resident rows.")

    if not education_records.empty:
        for col in ["attendance_rate", "progress_percent", "gpa_like_score"]:
            if col not in education_records.columns:
                education_records[col] = np.nan
            education_records[col] = pd.to_numeric(education_records[col], errors="coerce")

    if not health_records.empty:
        for col in ["nutrition_score", "sleep_score", "energy_score", "general_health_score"]:
            if col not in health_records.columns:
                health_records[col] = np.nan
            health_records[col] = pd.to_numeric(health_records[col], errors="coerce")

    base = residents.copy()
    base["resident_code"] = base["internal_code"].fillna("")
    base["present_age_num"] = base["present_age"].apply(parse_numeric_from_text)
    base["length_of_stay_num"] = base["length_of_stay"].apply(parse_numeric_from_text)
    base["label"] = base["reintegration_status"].apply(readiness_label)

    if not safehouses.empty:
        safehouse_map = safehouses.rename(columns={"name": "safehouse_name"})[["safehouse_id", "safehouse_code", "safehouse_name"]]
        base = base.merge(safehouse_map, on="safehouse_id", how="left")
    else:
        base["safehouse_code"] = "Unknown"
        base["safehouse_name"] = "Unknown"

    def agg_count(df, col_name):
        if df.empty:
            return pd.DataFrame({"resident_id": [], col_name: []})
        return df.groupby("resident_id").size().reset_index(name=col_name)

    def agg_mean(df, value_cols):
        if df.empty:
            data = {"resident_id": []}
            for c in value_cols:
                data[c] = []
            return pd.DataFrame(data)
        return df.groupby("resident_id")[value_cols].mean().reset_index()

    base = base.merge(agg_count(process_recordings, "process_recordings_count"), on="resident_id", how="left")
    base = base.merge(agg_count(home_visitations, "home_visitations_count"), on="resident_id", how="left")
    base = base.merge(agg_count(intervention_plans, "intervention_plans_count"), on="resident_id", how="left")
    base = base.merge(agg_count(incident_reports, "incident_reports_count"), on="resident_id", how="left")

    if not process_recordings.empty:
        p = process_recordings.copy()
        p["session_duration_minutes"] = pd.to_numeric(p["session_duration_minutes"], errors="coerce")
        p["progress_noted"] = p["progress_noted"].fillna(False).astype(int)
        p["concerns_flagged"] = p["concerns_flagged"].fillna(False).astype(int)
        p["referral_made"] = p["referral_made"].fillna(False).astype(int)
        p_means = p.groupby("resident_id")[["session_duration_minutes", "progress_noted", "concerns_flagged", "referral_made"]].mean().reset_index()
        base = base.merge(p_means, on="resident_id", how="left")

    if not home_visitations.empty:
        v = home_visitations.copy()
        v["follow_up_needed"] = v["follow_up_needed"].fillna(False).astype(int)
        v["safety_concerns_noted"] = v["safety_concerns_noted"].fillna(False).astype(int)
        v_means = v.groupby("resident_id")[["follow_up_needed", "safety_concerns_noted"]].mean().reset_index()
        base = base.merge(v_means, on="resident_id", how="left")

    if not education_records.empty:
        base = base.merge(agg_mean(education_records, ["attendance_rate", "progress_percent", "gpa_like_score"]), on="resident_id", how="left")

    if not health_records.empty:
        base = base.merge(agg_mean(health_records, ["nutrition_score", "sleep_score", "energy_score", "general_health_score"]), on="resident_id", how="left")

    if not incident_reports.empty:
        i = incident_reports.copy()
        i["resolved"] = i["resolved"].fillna(False).astype(int)
        i["high_severity"] = i["severity"].astype(str).str.lower().isin(["high", "critical", "severe"]).astype(int)
        i_means = i.groupby("resident_id")[["resolved", "high_severity"]].mean().reset_index()
        base = base.merge(i_means, on="resident_id", how="left")

    for col in ["process_recordings_count", "home_visitations_count", "intervention_plans_count", "incident_reports_count"]:
        if col not in base.columns:
            base[col] = 0
        base[col] = base[col].fillna(0)

    for col in FEATURE_COLUMNS:
        if col not in base.columns:
            base[col] = np.nan

    train_df = base.dropna(subset=["label"]).copy()
    train_df["label"] = train_df["label"].astype(int)
    return base, train_df


base, train_df = build_feature_tables(payload)
print("All residents (rows):", len(base), "| Labeled for training:", len(train_df))
print("Class balance (labeled):\n", train_df["label"].value_counts())



In [ ]:
# EDA: missingness & quick plots
labeled = train_df.copy()
print(labeled[FEATURE_COLUMNS].isna().mean().sort_values(ascending=False).head(12))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
train_df["label"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color=["#c44e52", "#55a868"])
axes[0].set_title("Label balance (0/1)")
axes[0].set_xlabel("readiness label")
sample_num = pd.to_numeric(train_df["present_age_num"], errors="coerce")
sample_num.dropna().plot(kind="hist", ax=axes[1], bins=10, color="#4c72b0")
axes[1].set_title("Distribution: present_age_num (labeled train pool)")
plt.tight_layout()
plt.show()



## Phase 3: Data Preparation

- **Features:** `FEATURE_COLUMNS` (categorical + numeric) — preprocessing uses `ColumnTransformer` with imputation + scaling / one-hot encoding **inside** sklearn `Pipeline` so test data does not leak statistics from the test fold into the training preprocessor (the preprocessor is fit on `X_train` only).
- **Train/test split:** 75% / 25%, `random_state=42`, stratified when possible (for baseline comparison and permutation importance).
- **Production evaluation** uses **stratified k-fold CV** on the full labeled pool plus **OOF threshold tuning**; see Phase 5.


In [ ]:
MIN_LABELED = 8
if len(train_df) < MIN_LABELED:
    raise ValueError(f"Need at least {MIN_LABELED} labeled rows; got {len(train_df)}")
if train_df["label"].nunique() < 2:
    raise ValueError("Need both classes in labeled data.")

X = train_df[FEATURE_COLUMNS].copy()
y = train_df["label"].astype(int)
stratify_y = y if y.nunique() > 1 else None
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=stratify_y
    )
except ValueError:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print("Train:", len(X_train), "Test:", len(X_test))



## Phase 4: Modeling

- **Predictive:** `RandomForestClassifier(..., class_weight="balanced_subsample")` (same hyperparameters as `girls_reintegration_train.py`)
- **Explanatory:** `LogisticRegression(max_iter=2000, class_weight="balanced")`

Each uses the same preprocessing pipeline fitted on `X_train` for the initial fit; the evaluation cell refits the forest on **all** labeled rows for deployment-aligned scoring.


In [ ]:
# Share one preprocessor instance across both pipelines (same as production script)
pre = build_preprocessor(X_train)
predictive = Pipeline(
    [
        ("pre", pre),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                min_samples_leaf=3,
                random_state=42,
                class_weight="balanced_subsample",
            ),
        ),
    ]
)
explanatory = Pipeline(
    [("pre", pre), ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))]
)

predictive.fit(X_train, y_train)
explanatory.fit(X_train, y_train)
print("Fitted RandomForest + LogisticRegression pipelines (shared preprocessor).")



## Phase 5: Evaluation

Compare the Random Forest to a **baseline** (`DummyClassifier(strategy="most_frequent")`) on the **same** `X_test, y_test`. Report ROC-AUC (if both classes present), Average Precision, F1.


In [ ]:
def metric_pack(y_true, y_prob, y_pred, n_train, n_test):
    roc = float(roc_auc_score(y_true, y_prob)) if len(set(y_true)) > 1 else None
    return {
        "roc_auc": roc,
        "avg_precision": float(average_precision_score(y_true, y_prob)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "train_rows": n_train,
        "test_rows": n_test,
    }


prob_test = predictive.predict_proba(X_test)[:, 1]
pred_test = (prob_test >= 0.5).astype(int)
metrics_rf = metric_pack(y_test, prob_test, pred_test, len(X_train), len(X_test))

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)
base_prob = baseline.predict_proba(X_test)[:, 1] if baseline.predict_proba(X_test).shape[1] > 1 else np.full(len(y_test), float(np.mean(y_train)))
metrics_base = metric_pack(y_test, base_prob, base_pred, len(X_train), len(X_test))

print("=== Random Forest (test) ===")
print(metrics_rf)
print("\n=== Majority-class baseline (test) ===")
print(metrics_base)

imp = permutation_importance(predictive, X_test, y_test, n_repeats=10, random_state=42)
imp_df = pd.DataFrame({"feature": X_test.columns, "importance": imp.importances_mean}).sort_values("importance", ascending=False).head(10)
print("\n=== Top permutation importances (RandomForest) ===")
print(imp_df.to_string(index=False))


def rf_template_for(X_df: pd.DataFrame) -> Pipeline:
    return Pipeline(
        [
            ("pre", build_preprocessor(X_df)),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    min_samples_leaf=3,
                    random_state=42,
                    class_weight="balanced_subsample",
                ),
            ),
        ]
    )


def _cv_fold_count(ys: pd.Series, max_folds: int = 5) -> int:
    n = len(ys)
    if n < 8:
        return 0
    vc = ys.value_counts()
    if len(vc) < 2:
        return 0
    min_class = int(vc.min())
    n_splits = min(max_folds, min_class, max(2, n // 2))
    n_splits = max(2, int(n_splits))
    if n < n_splits * 2:
        n_splits = min(n_splits, n // 2)
    return n_splits if n_splits >= 2 else 0


def best_f1_threshold(y_true, proba):
    best_t, best_f1 = 0.5, 0.0
    for t in np.linspace(0.05, 0.95, 19):
        pred = (proba >= t).astype(int)
        f1v = f1_score(y_true, pred, zero_division=0)
        if f1v > best_f1:
            best_f1, best_t = f1v, t
    return float(best_t), float(best_f1)


n_sp = _cv_fold_count(y, 5)
if n_sp >= 2:
    tmpl = rf_template_for(X)
    skf = StratifiedKFold(n_splits=n_sp, shuffle=True, random_state=42)
    oof_proba = cross_val_predict(clone(tmpl), X, y, cv=skf, method="predict_proba", n_jobs=1)[:, 1]
    tau_star, f1_oof = best_f1_threshold(y.values, oof_proba)
    print("\n=== Stratified CV + OOF threshold (aligned with production train script) ===")
    print({"cv_folds": n_sp, "optimal_threshold": tau_star, "f1_at_optimal_threshold_oof": f1_oof})
else:
    print("\n(Stratified CV skipped: too few labeled rows.)")

predictive = rf_template_for(X)
predictive.fit(X, y)
print("Refit predictive pipeline on full labeled pool (matches deployment scoring).")



In [ ]:
# Score all residents for worklist-style summary
X_all = base[FEATURE_COLUMNS].copy()
base = base.copy()
base["readiness_score"] = predictive.predict_proba(X_all)[:, 1]
base["readiness_band"] = pd.cut(
    base["readiness_score"], bins=[-0.001, 0.35, 0.65, 1.0], labels=["Low", "Medium", "High"]
).astype(str)
print("Readiness band counts:\n", base["readiness_band"].value_counts())
display_cols = ["resident_id", "resident_code", "readiness_score", "readiness_band"]
print(base.sort_values("readiness_score", ascending=False)[display_cols].head(12).to_string(index=False))



## Phase 6: Deployment

**Production (this repo):** Training is **not** executed from this notebook in deployment. The ASP.NET host runs `backend/ML/girls_reintegration_train.py` with a JSON payload from the database (`GirlsReintegrationMlPipelineService`), writes **`backend/ML/artifacts/girls_reintegration_mlr_latest.json`**, and the admin UI reads insights via **`GET /api/ml/girls-reintegration/insights`**. Trigger training with **`POST /api/ml/girls-reintegration/train`** (admin).

**This notebook:** For coursework, we optionally serialize the **fitted sklearn Pipeline** with `joblib` for reproducibility. The live app consumes **JSON metrics/worklist**, not the pickle.


In [ ]:
# Optional: save fitted pipeline (course template / local reproducibility)
_art = _CWD / "ml-pipelines" / "artifacts"
if not _art.is_dir():
    _art = _CWD / "artifacts"
_art.mkdir(parents=True, exist_ok=True)
_out = _art / "girls_reintegration_rf_pipeline.joblib"
joblib.dump(predictive, _out)
print("Saved:", _out.resolve())

